In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

!pip install mne
import mne
import numpy as np
import os

In [ ]:
# ==== SET THIS ====
eeglab_path = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData/3/3_001.set'  # update to your real file path

# ==== LOAD EEG ====
raw = mne.io.read_raw_eeglab(eeglab_path, preload=True)
print(f"Loaded file with {len(raw.annotations)} annotations.")

# ==== DETECT LONG GAPS BETWEEN EVENTS ====
onsets = raw.annotations.onset
descriptions = raw.annotations.description

# Compute time gaps between successive annotations
gaps = np.diff(onsets)
threshold = 100.0  # seconds
long_gap_indices = np.where(gaps > threshold)[0]

print(f"Found {len(long_gap_indices)} long gaps > {threshold}s")

# Annotate long gaps
for idx in long_gap_indices:
    onset = onsets[idx]
    duration = gaps[idx]
    raw.annotations.append(onset, duration, 'BAD_long_gap')


# ==== PLOT TO VISUALLY INSPECT ====
raw.plot(duration=30, n_channels=30, scalings='auto')


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Import libraries
import os
import mne
import numpy as np
import pandas as pd

# Step 3: Set paths and parameters
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'
gap_threshold = 100.0  # seconds
summary_records = []

# Step 4: Loop through participant folders
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

for pid in participants:
    set_path = os.path.join(root_dir, pid, f"{pid}_001.set")
    fif_path = os.path.join(root_dir, pid, f"{pid}_annotated_raw.fif")

    if not os.path.exists(set_path):
        print(f"[!] {pid}: .set file not found.")
        continue

    print(f"\n=== Processing Participant {pid} ===")

    try:
        raw = mne.io.read_raw_eeglab(set_path, preload=True)
    except Exception as e:
        print(f"[ERROR loading {pid}]: {e}")
        continue

    # Step 5: Detect long gaps between annotations
    onsets = raw.annotations.onset
    gaps = np.diff(onsets)
    long_gap_indices = np.where(gaps > gap_threshold)[0]

    print(f"→ Found {len(long_gap_indices)} long gaps > {gap_threshold}s")

    for idx in long_gap_indices:
        onset = onsets[idx]
        duration = gaps[idx]
        raw.annotations.append(onset, duration, 'break')
        summary_records.append({
            'participant': pid,
            'onset_sec': round(onset, 3),
            'duration_sec': round(duration, 3)
        })

    # Step 6: Save annotated raw file
    raw.save(fif_path, overwrite=True)
    print(f"✓ Saved annotated raw to: {fif_path}")

# Step 7: Save CSV summary of breaks
summary_df = pd.DataFrame(summary_records)
csv_output_path = os.path.join(root_dir, 'break_summary.csv')
summary_df.to_csv(csv_output_path, index=False)
print(f"\n📁 Summary CSV saved to: {csv_output_path}")

# Show first few rows
summary_df.head()


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=summary_df)

In [ ]:
import os
import mne
import numpy as np

# === PATH TO EEGData FOLDER ===
root_dir = '/content/drive/MyDrive/2025_Summer (Distal Speech Rate)/Project_Code/EEGData'

# === PARAMETERS ===
gap_threshold = 100.0  # seconds

# === LOOP THROUGH PARTICIPANTS ===
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

for pid in participants:
    set_path = os.path.join(root_dir, pid, f"{pid}_001.set")

    if not os.path.exists(set_path):
        print(f"[!] {pid}: .set file not found.")
        continue

    print(f"\n=== Processing Participant {pid} ===")

    # Load raw EEG
    try:
        raw = mne.io.read_raw_eeglab(set_path, preload=True)
    except Exception as e:
        print(f"[ERROR loading {pid}]: {e}")
        continue

    # Extract annotations
    onsets = raw.annotations.onset
    descriptions = raw.annotations.description
    gaps = np.diff(onsets)

    long_gap_indices = np.where(gaps > gap_threshold)[0]
    print(f"→ Found {len(long_gap_indices)} long gaps > {gap_threshold}s")

    # Annotate long gaps as break
    for idx in long_gap_indices:
        onset = onsets[idx]
        duration = gaps[idx]
        raw.annotations.append(onset, duration, 'break')

    # Optional: print first few new annotations
    for ann in raw.annotations:
        if ann['description'] == 'BAD_long_gap':
            print(f"   BAD @ {ann['onset']:.1f}s ({ann['duration']:.1f}s)")

print("\n✅ All participants processed.")
